# CV Intelligent — Backend IA (CrewAI + OpenRouter)


## 1. Installation


In [64]:
!pip install crewai pdfplumber python-dotenv pydantic litellm -q


D�finition de macro non valide.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\rayan\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


## 2. Configuration


In [65]:
import os, sys, json, re
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List, Optional
from crewai import Agent, Task, Crew, Process, LLM
import pdfplumber

sys.setrecursionlimit(10000)
load_dotenv()

modele = LLM(
    model="openrouter/google/gemma-3-27b-it:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1500
)
print("Modele IA OpenRouter charge.")


Modele IA OpenRouter charge.


## 3. Modèles de Données


In [66]:
class Experience(BaseModel):
    titre: Optional[str] = None
    entreprise: Optional[str] = None
    duree: Optional[str] = None
    lieu: Optional[str] = None
    description: Optional[List[str]] = []

class Formation(BaseModel):
    diplome: Optional[str] = None
    etablissement: Optional[str] = None
    annee: Optional[str] = None
    description: Optional[List[str]] = []

class Projet(BaseModel):
    nom: Optional[str] = None
    description: Optional[List[str]] = []
    technologies: Optional[List[str]] = []

class GroupeCompetence(BaseModel):
    categorie: str
    elements: List[str]

class ProfilCandidat(BaseModel):
    prenom: Optional[str] = None
    nom: Optional[str] = None
    email: Optional[str] = None
    telephone: Optional[str] = None
    ville: Optional[str] = None
    linkedin: Optional[str] = None
    github: Optional[str] = None
    portfolio: Optional[str] = None
    resume: Optional[str] = None
    experiences: Optional[List[Experience]] = []
    formations: Optional[List[Formation]] = []
    competences: Optional[List[GroupeCompetence]] = []
    projets: Optional[List[Projet]] = []
    langues: Optional[List[str]] = []

class AnalyseOffre(BaseModel):
    titre_poste: Optional[str] = None
    entreprise: Optional[str] = None
    competences_requises: Optional[List[str]] = []
    competences_souhaitees: Optional[List[str]] = []
    mots_cles_ats: Optional[List[str]] = []
    resume: Optional[str] = None

class ResultatFinal(BaseModel):
    profil_optimise: Optional[ProfilCandidat] = None
    score_ats: Optional[int] = None
    commentaire_ats: Optional[str] = None
    lettre_motivation: Optional[str] = None

print("Modeles de donnees prets.")


Modeles de donnees prets.


## 4. Extraction PDF


In [67]:
def lire_texte_depuis_pdf(chemin: str) -> str:
    texte_complet = ""
    with pdfplumber.open(chemin) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                texte_complet += t + "\n"
    return texte_complet.strip()

print("Extraction PDF prete.")


Extraction PDF prete.


## 5. Agents IA


In [68]:
agent_extracteur = Agent(
    role="Expert Analyseur de CV",
    goal="Extraire toutes les informations du CV en JSON",
    backstory="Parseur meticuleux. Tu n'inventes rien.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

agent_analyseur_offre = Agent(
    role="Analyste en Recrutement",
    goal="Analyser les offres d'emploi pour mots-cles ATS",
    backstory="Tu identifies les competences requises.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

agent_redacteur_cv = Agent(
    role="Redacteur de CV",
    goal="Adapter le CV en JSON par rapport a l'offre",
    backstory="Tu reformules les experiences. JSON uniquement.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

agent_relecteur = Agent(
    role="Auditeur Qualite CV",
    goal="Verifier qu'aucune info vitale n'a ete perdue",
    backstory="JSON final parfait sans rien inventer.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

agent_optimiseur_ats = Agent(
    role="Specialiste ATS",
    goal="Evaluer la compatibilite ATS (0-100)",
    backstory="JSON avec score_ats et commentaire_ats.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

agent_redacteur_lettre = Agent(
    role="Specialiste Lettre de Motivation",
    goal="Rediger une lettre de motivation max 250 mots",
    backstory="Lettre professionnelle et percutante.",
    llm=modele, verbose=False, max_iter=2, max_rpm=2
)

print("Agents IA prets.")


Agents IA prets.


## 6. Taches


In [69]:
def creer_taches(texte_cv: str, texte_offre: str):
    texte_cv_tronque = texte_cv[:3000]
    schema_profil = json.dumps(ProfilCandidat.model_json_schema(), ensure_ascii=False)
    schema_offre  = json.dumps(AnalyseOffre.model_json_schema(), ensure_ascii=False)

    tache_extraction = Task(
        description=(
            "CV brut:\n" + texte_cv_tronque +
            "\n\nExtrais tout en JSON selon ce schema:\n" + schema_profil
        ),
        expected_output="JSON valide ProfilCandidat",
        agent=agent_extracteur
    )

    tache_analyse_offre = Task(
        description=(
            "Offre d'emploi:\n" + texte_offre +
            "\n\nAnalyse en JSON selon ce schema:\n" + schema_offre
        ),
        expected_output="JSON valide AnalyseOffre",
        agent=agent_analyseur_offre
    )

    tache_redaction_cv = Task(
        description=(
            "Adapte le CV pour l'offre. Retourne UNIQUEMENT un JSON:\n" + schema_profil
        ),
        expected_output="JSON ProfilCandidat optimise",
        agent=agent_redacteur_cv,
        context=[tache_extraction, tache_analyse_offre]
    )

    tache_relecture = Task(
        description=(
            "Verifie aucune perte de donnees. Retourne JSON final:\n" + schema_profil
        ),
        expected_output="JSON final ProfilCandidat",
        agent=agent_relecteur,
        context=[tache_extraction, tache_redaction_cv]
    )

    tache_ats = Task(
        description='Score ATS (0-100). Retourne JSON: {"score_ats": int, "commentaire_ats": str}',
        expected_output="JSON score ATS",
        agent=agent_optimiseur_ats,
        context=[tache_relecture, tache_analyse_offre]
    )

    tache_lettre = Task(
        description="Redige une lettre de motivation en 250 mots maximum. Texte uniquement, pas de JSON.",
        expected_output="Texte lettre motivation",
        agent=agent_redacteur_lettre,
        context=[tache_relecture, tache_analyse_offre]
    )

    return [tache_extraction, tache_analyse_offre, tache_redaction_cv, tache_relecture, tache_ats, tache_lettre]

print("Taches definies.")


Taches definies.


## 7. Pipeline Principal


In [70]:
def extraire_json(texte: str) -> dict:
    texte = texte.strip()
    match = re.search(r'```(?:json)?\s*({.*?})\s*```', texte, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    match = re.search(r'({.*})', texte, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    return json.loads(texte)


def generer_cv(chemin_cv_pdf: str, texte_offre: str) -> ResultatFinal:
    print("[1/5] Lecture du CV PDF...")
    texte_cv = lire_texte_depuis_pdf(chemin_cv_pdf)
    if not texte_cv:
        raise ValueError("Le PDF est vide ou illisible.")

    print("[2/5] Creation des taches...")
    taches = creer_taches(texte_cv, texte_offre)

    print("[3/5] Lancement du pipeline IA (CrewAI + OpenRouter)...")
    crew = Crew(
        agents=[
            agent_extracteur,
            agent_analyseur_offre,
            agent_redacteur_cv,
            agent_relecteur,
            agent_optimiseur_ats,
            agent_redacteur_lettre
        ],
        tasks=taches,
        process=Process.sequential,
        verbose=False
    )
    resultats = crew.kickoff()

    print("[4/5] Traitement des resultats...")
    sorties = resultats.tasks_output

    profil_optimise  = ProfilCandidat(**extraire_json(sorties[3].raw))
    donnees_ats      = extraire_json(sorties[4].raw)
    score_ats        = int(donnees_ats.get("score_ats", 0))
    commentaire_ats  = donnees_ats.get("commentaire_ats", "")
    lettre_motivation = sorties[5].raw.strip()

    print("[5/5] Pipeline termine avec succes.")
    return ResultatFinal(
        profil_optimise=profil_optimise,
        score_ats=score_ats,
        commentaire_ats=commentaire_ats,
        lettre_motivation=lettre_motivation
    )

print("Pipeline pret.")


Pipeline pret.


## 8. Test


In [71]:
CHEMIN_CV = r"../CV_Rayane_Berrada.pdf"

OFFRE = """
Titre : Developpeur Full Stack Junior
Entreprise : TechCorp

Nous recherchons un developpeur Full Stack motivé maitrisant React, Python et FastAPI.
Experience avec les bases de donnees MongoDB appreciee.
Connaissances en IA et integration d'API un plus.
Bon niveau de communication, esprit d'equipe, autonomie.
"""

resultat = generer_cv(CHEMIN_CV, OFFRE)

print("\n" + "=" * 60)
print(f"Score ATS : {resultat.score_ats}/100")
print(f"Commentaire : {resultat.commentaire_ats}")
print("=" * 60)
print("\nProfil optimise (JSON):")
print(json.dumps(resultat.profil_optimise.model_dump(), ensure_ascii=False, indent=2))
print("\nLettre de motivation:")
print(resultat.lettre_motivation)


[1/5] Lecture du CV PDF...
[2/5] Creation des taches...
[3/5] Lancement du pipeline IA (CrewAI + OpenRouter)...


RateLimitError: litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations","provider_name":"Google AI Studio","is_byok":false}},"user_id":"user_3C4fc5h8zHRvrGEYcQuCpNG7nBn"}

## 9. Export JSON (optionnel)


In [ ]:
def sauvegarder_resultat(resultat: ResultatFinal, dossier: str = "."):
    os.makedirs(dossier, exist_ok=True)

    with open(f"{dossier}/cv_optimise.json", "w", encoding="utf-8") as f:
        json.dump(resultat.profil_optimise.model_dump(), f, ensure_ascii=False, indent=2)

    with open(f"{dossier}/lettre_motivation.txt", "w", encoding="utf-8") as f:
        f.write(resultat.lettre_motivation)

    with open(f"{dossier}/score_ats.json", "w", encoding="utf-8") as f:
        json.dump({"score_ats": resultat.score_ats, "commentaire": resultat.commentaire_ats}, f, ensure_ascii=False, indent=2)

    print(f"Fichiers sauvegardes dans : {dossier}")


sauvegarder_resultat(resultat, dossier="output")
